In [1]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

TensorFlow version:  2.17.0


In [2]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project_holdout.yaml"
VAL_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch11_margin_15_wb_alif_clahe/project.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [3]:
model = YOLO(WEIGHTS_PTH)

In [4]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

# inferer_adapter = TFLiteInferenceWithInputAdapter(
#     yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
#     head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
#     adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
#     conf_th=CONF_TH,
#     iou_th=IOU_TH,
#     imgsz=320,
# )

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
args["data"] = VAL_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch11_margin_15_wb_alif_clahe/labels/val.cache... 732 images, 70 backgrounds, 0 corrupt: 100%|██████████| 732/732 [00:00<?, ?it/s]


tst validation: 100%|██████████| 6/6 [00:30<00:00,  5.01s/it]


                   all        732       2137      0.997      0.993      0.995      0.883      0.998      0.994      0.995      0.995
   task    F1     P     R  MAP@50
0   box 0.995 0.997 0.993   0.995
1  pose 0.996 0.998 0.994   0.995


In [6]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.22s/it]


                   all        148        409      0.935      0.875      0.937      0.693      0.951       0.89      0.959      0.954
   task    F1     P     R  MAP@50
0   box 0.904 0.935 0.875   0.937
1  pose 0.919 0.951 0.890   0.959


In [7]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]
tst validation:   0%|          | 0/2 [00:00<?, ?it/s]

tst validation: 100%|██████████| 2/2 [00:08<00:00,  4.06s/it]


                   all        191        417      0.922      0.677      0.816      0.547      0.945      0.694      0.839      0.837
   task    F1     P     R  MAP@50
0   box 0.781 0.922 0.677   0.816
1  pose 0.800 0.945 0.694   0.839
